# WriteWise — Stage 1: MobileNetV2 Fine-Tuning on CCC

Fine-tunes MobileNetV2 (ImageNet-pretrained) on the CCC cursive character dataset (57,293 characters across 52 classes) to learn cursive-specific stroke features. See `ML_PIPELINE.md §4` and `PRD.md §11`.

**This model is never deployed as-is** — only its convolutional backbone carries forward into Stage 2's letter-formation regression head.

## Hardware Requirement
- Colab runtime set to **T4 GPU** (Runtime → Change runtime type → T4 GPU)


## 1. Setup & Environment


In [ ]:
# Mount Google Drive for data access and persistent checkpoint saving
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, zipfile, shutil
from pathlib import Path

# Configuration — set checkpoint directory in Drive
DRIVE_DIR = '/content/drive/MyDrive/writewise/training'
CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# High-performance local caching on Colab local SSD (/content/data/processed):
LOCAL_DATA_DIR = '/content/data/processed'

# Look for processed.zip in common locations (Drive or uploaded directly to Colab)
candidate_zip_paths = [
    '/content/processed.zip',
    f'{DRIVE_DIR}/data/processed.zip',
    f'{DRIVE_DIR}/processed.zip',
    '/content/drive/MyDrive/writewise/data/processed.zip',
    '/content/drive/MyDrive/writewise/processed.zip',
    '/content/drive/MyDrive/processed.zip',
]

zip_found = None
for p in candidate_zip_paths:
    if os.path.exists(p):
        zip_found = p
        break

if zip_found and not (os.path.exists(f'{LOCAL_DATA_DIR}/train/images.npy') or os.path.exists(f'{LOCAL_DATA_DIR}/train')):
    print(f'Extracting {zip_found} to Colab local SSD ({LOCAL_DATA_DIR}) for ultra-fast training...')
    with zipfile.ZipFile(zip_found, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA_DIR)
    DATA_DIR = LOCAL_DATA_DIR
elif os.path.exists(LOCAL_DATA_DIR) and (os.path.exists(f'{LOCAL_DATA_DIR}/train/images.npy') or os.path.exists(f'{LOCAL_DATA_DIR}/train')):
    DATA_DIR = LOCAL_DATA_DIR
elif os.path.exists(f'{DRIVE_DIR}/data/processed'):
    DATA_DIR = f'{DRIVE_DIR}/data/processed'
else:
    found_dirs = glob.glob('/content/drive/MyDrive/**/processed', recursive=True) + glob.glob('/content/drive/MyDrive/**/train', recursive=True)
    if found_dirs:
        DATA_DIR = str(Path(found_dirs[0]).parent) if 'train' in found_dirs[0] else found_dirs[0]
    else:
        DATA_DIR = LOCAL_DATA_DIR

print(f'Active DATA_DIR:       {DATA_DIR}')
print(f'Active CHECKPOINT_DIR: {CHECKPOINT_DIR}')


In [ ]:
import numpy as np
import tensorflow as tf

print(f'TensorFlow version: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU available: {len(gpus) > 0}')
if gpus:
    print(f'Using GPU device: {gpus[0].name}')
else:
    print('WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU.')


## 2. Load Dataset (Train & Val Splits)


In [ ]:
def load_split(split_dir: str) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Load dataset split as compact uint8 arrays (prevents Colab RAM crashes)."""
    split_path = Path(split_dir)
    images_file = split_path / 'images.npy'
    labels_file = split_path / 'labels.npy'
    
    if images_file.exists() and labels_file.exists():
        print(f'  Loading pre-packaged uint8 matrices from {split_path}...')
        images = np.load(images_file)  # uint8: ~316 MB for train, not 12 GB!
        raw_labels = np.load(labels_file)
        labels = [str(l) for l in raw_labels]
    else:
        print(f'  Loading individual .npy files from {split_path}...')
        images = []
        labels = []
        for npy_file in sorted(split_path.glob('*.npy')):
            if npy_file.name in ('images.npy', 'labels.npy'):
                continue
            img = np.load(npy_file)
            label = npy_file.stem.split('_')[0]
            images.append(img)
            labels.append(label)
        images = np.array(images, dtype=np.uint8)
    
    # Encode labels to integers (52 classes: 26 lower + 26 upper)
    class_names = sorted(set(labels))
    label_to_idx = {name: idx for idx, name in enumerate(class_names)}
    labels_encoded = np.array([label_to_idx[l] for l in labels], dtype=np.int32)
    
    return images, labels_encoded, class_names

print('Loading training data...')
X_train, y_train, class_names = load_split(f'{DATA_DIR}/train')
print(f'  Train: {X_train.shape} (dtype: {X_train.dtype}, {X_train.nbytes / 1e6:.1f} MB in RAM), {len(class_names)} classes')

print('Loading validation data...')
X_val, y_val, _ = load_split(f'{DATA_DIR}/val')
print(f'  Val:   {X_val.shape} (dtype: {X_val.dtype}, {X_val.nbytes / 1e6:.1f} MB in RAM)')

NUM_CLASSES = len(class_names)
print(f'\nTotal Classes ({NUM_CLASSES}): {class_names}')


## 3. Streaming Preprocessing & Data Augmentation (ML_PIPELINE §4.2)

**RAM-Safe Streaming Pipeline:** Images stay as raw `uint8` in memory (~300MB total).
Conversion to 3-channel RGB, float32 normalization `[-1, 1]`, and rotation augmentation
occur on-the-fly per batch during training using `tf.data.AUTOTUNE`.


In [ ]:
INPUT_SIZE = 96  # ML_PIPELINE §2.3
BATCH_SIZE = 32

# ML_PIPELINE §4.2: rotation capped at ±15°, slight zoom/translate
augmentation_layer = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(
        factor=15/360,  # ±15 degrees ceiling
        fill_mode='constant',
        fill_value=-1.0,  # white background in [-1, 1] range
    ),
    tf.keras.layers.RandomZoom(
        height_factor=(-0.1, 0.1),
        width_factor=(-0.1, 0.1),
        fill_mode='constant',
        fill_value=-1.0,
    ),
    tf.keras.layers.RandomTranslation(
        height_factor=0.05,
        width_factor=0.05,
        fill_mode='constant',
        fill_value=-1.0,
    ),
])

@tf.function
def transform_sample(image, label):
    """Streamed on-the-fly conversion from (96, 96) uint8 to (96, 96, 3) float32 in [-1, 1]."""
    # Grayscale -> 3-channel (MobileNetV2 RGB requirement)
    image = tf.expand_dims(image, -1)
    image = tf.repeat(image, 3, axis=-1)
    # Normalize to [-1, 1]
    image = (tf.cast(image, tf.float32) / 127.5) - 1.0
    # One-hot encode label
    label_onehot = tf.one_hot(label, NUM_CLASSES)
    return image, label_onehot

def build_dataset(images: np.ndarray, labels: np.ndarray, augment: bool = False):
    """Build a memory-efficient tf.data.Dataset using on-the-fly batch transformation."""
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if augment:
        ds = ds.shuffle(buffer_size=2000, seed=42)
    
    # Streaming transform (only processes 32 samples at a time in RAM)
    ds = ds.map(transform_sample, num_parallel_calls=tf.data.AUTOTUNE)
    
    if augment:
        ds = ds.map(lambda x, y: (augmentation_layer(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

print('Building streaming datasets...')
train_ds = build_dataset(X_train, y_train, augment=True)
val_ds = build_dataset(X_val, y_val, augment=False)

for batch_x, batch_y in train_ds.take(1):
    print(f'  Sample batch shape: {batch_x.shape} (dtype: {batch_x.dtype}), Labels: {batch_y.shape}')
print('Datasets ready with minimal RAM consumption!')


## 4. MobileNetV2 Architecture (ML_PIPELINE §3)


In [ ]:
# ImageNet-pretrained MobileNetV2 base
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(INPUT_SIZE, INPUT_SIZE, 3),
    include_top=False,
    weights='imagenet',
)

# Classification head
x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs=base_model.input, outputs=output)

print(f'Base model layers: {len(base_model.layers)}')
print(f'Total model parameters: {model.count_params():,}')


## 5. Phase A — Head-Only Training (ML_PIPELINE §4.1)

Freeze the entire MobileNetV2 base. Train only the classification head for 10 epochs
to let it adapt without disturbing pretrained ImageNet weights.


In [ ]:
base_model.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print('Phase A: Head-only training (base frozen)')
print(f'  Trainable parameters: {sum(p.numpy().size for p in model.trainable_weights):,}')

history_a = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
)

print(f'\nPhase A complete.')
print(f'  Final train accuracy: {history_a.history["accuracy"][-1]:.4f}')
print(f'  Final val accuracy:   {history_a.history["val_accuracy"][-1]:.4f}')


## 6. Phase B — Partial Unfreeze & Fine-Tuning (ML_PIPELINE §4.1, §4.3)

Unfreeze top ~30% of MobileNetV2 layers and continue training at a lower learning rate (1e-5).
Early stopping on validation loss with checkpoint saving.


In [ ]:
base_model.trainable = True
total_layers = len(base_model.layers)
freeze_until = int(total_layers * 0.7)

for layer in base_model.layers[:freeze_until]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfroze {trainable_count}/{total_layers} base layers')

# Recompile with lower learning rate (ML_PIPELINE §4.3)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print(f'Total trainable parameters: {sum(p.numpy().size for p in model.trainable_weights):,}')

# Callbacks
checkpoint_path = f'{CHECKPOINT_DIR}/stage1_best.keras'
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

print('\nPhase B: Fine-tuning with early stopping')
history_b = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
)

print(f'\nPhase B complete.')
print(f'  Best val loss:     {min(history_b.history["val_loss"]):.4f}')
print(f'  Best val accuracy: {max(history_b.history["val_accuracy"]):.4f}')
print(f'  Checkpoint saved:  {checkpoint_path}')


## 7. Training Summary & Curves


In [ ]:
import matplotlib.pyplot as plt

all_acc = history_a.history['accuracy'] + history_b.history['accuracy']
all_val_acc = history_a.history['val_accuracy'] + history_b.history['val_accuracy']
all_loss = history_a.history['loss'] + history_b.history['loss']
all_val_loss = history_a.history['val_loss'] + history_b.history['val_loss']
phase_a_epochs = len(history_a.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(all_acc, label='Train')
ax1.plot(all_val_acc, label='Validation')
ax1.axvline(x=phase_a_epochs - 0.5, color='gray', linestyle='--', alpha=0.6, label='Phase A -> B')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss plot
ax2.plot(all_loss, label='Train')
ax2.plot(all_val_loss, label='Validation')
ax2.axvline(x=phase_a_epochs - 0.5, color='gray', linestyle='--', alpha=0.6, label='Phase A -> B')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
curves_path = f'{CHECKPOINT_DIR}/training_curves.png'
plt.savefig(curves_path, dpi=150)
plt.show()

print(f'\nFinal training results:')
print(f'  Best validation accuracy: {max(all_val_acc):.4f}')
print(f'  Best validation loss:     {min(all_val_loss):.4f}')
print(f'  Curves saved to:          {curves_path}')


## 8. Final Evaluation on Held-Out Test Set (PRD §11 & ML_PIPELINE §5)

Evaluates the best checkpoint (`stage1_best.keras`) against CCC's untouched 19,133-character test set.
Computes Overall Accuracy (target: >=90%), Macro Precision, Recall, F1-Score, and generates the confusion matrix.


In [ ]:
import json
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

eval_dir = f'{CHECKPOINT_DIR}/evaluation'
os.makedirs(eval_dir, exist_ok=True)

print('Loading best model checkpoint for evaluation...')
eval_model = tf.keras.models.load_model(checkpoint_path)

print('Loading untouched CCC test set...')
X_test, y_test, test_classes = load_split(f'{DATA_DIR}/test')
print(f'  Test samples: {len(X_test)}')
print(f'  Classes:      {len(test_classes)}')

# RAM-safe streamed test dataset (no massive float32 arrays in RAM!)
@tf.function
def test_preprocess(image):
    image = tf.expand_dims(image, -1)
    image = tf.repeat(image, 3, axis=-1)
    return (tf.cast(image, tf.float32) / 127.5) - 1.0

test_ds = tf.data.Dataset.from_tensor_slices(X_test)
test_ds = test_ds.map(test_preprocess, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)

print('Running test set inference...')
test_predictions = eval_model.predict(test_ds, verbose=1)
y_pred = np.argmax(test_predictions, axis=1)

# Metrics
acc = float(accuracy_score(y_test, y_pred))
prec = float(precision_score(y_test, y_pred, average='macro', zero_division=0))
rec = float(recall_score(y_test, y_pred, average='macro', zero_division=0))
f1 = float(f1_score(y_test, y_pred, average='macro', zero_division=0))

passed = acc >= 0.90

print()
print('=' * 65)
print('  STAGE 1 FINAL EVALUATION METRICS (CCC TEST SET - 19,133 SAMPLES)')
print('=' * 65)
print(f'  Overall Accuracy:  {acc:.4f} ({acc*100:.2f}%)')
print(f'  Macro Precision:   {prec:.4f}')
print(f'  Macro Recall:      {rec:.4f}')
print(f'  Macro F1-Score:    {f1:.4f}')
print('  PRD §11 Target:    >= 90.0%')
print(f'  Status:            {"✅ PASS (Target Achieved!)" if passed else "❌ BELOW TARGET"}')
print('=' * 65)

# Save Classification Report
report = classification_report(y_test, y_pred, target_names=test_classes, zero_division=0)
with open(f'{eval_dir}/classification_report.txt', 'w', encoding='utf-8') as f:
    f.write('Stage 1 Evaluation — Classification Report\n')
    f.write(f'Accuracy: {acc:.4f} | F1: {f1:.4f}\n\n')
    f.write(report)

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
np.savetxt(f'{eval_dir}/confusion_matrix.csv', cm, delimiter=',', fmt='%d')

plt.figure(figsize=(16, 14))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title(f'Stage 1 Confusion Matrix (Test Accuracy: {acc*100:.1f}%)', fontsize=14)
plt.colorbar()
ticks = list(range(len(test_classes)))
plt.xticks(ticks, test_classes, rotation=90, fontsize=6)
plt.yticks(ticks, test_classes, fontsize=6)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig(f'{eval_dir}/confusion_matrix.png', dpi=150)
plt.show()

# Save summary JSON
summary = {
    'checkpoint': checkpoint_path,
    'test_samples': len(X_test),
    'accuracy': round(acc, 6),
    'macro_precision': round(prec, 6),
    'macro_recall': round(rec, 6),
    'macro_f1': round(f1, 6),
    'passes_prd_target': passed
}
with open(f'{eval_dir}/summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print(f'\nAll artifacts saved to Drive at: {eval_dir}')
